# AeroVision Lab — frozen held-out test-dev evaluation
Import into a **private** Kaggle notebook. Attach the same private full-training
results Dataset; enable Internet and T4 GPU. Enable Files-only persistence.
No training, threshold search or model selection occurs here. This runs the
saved PyTorch checkpoint, not the ONNX model. The ONNX checks were a separate
small CPU parity test.

Frozen candidate: whole-image 1280, confidence 0.30 for operating-point metrics,
NMS IoU 0.7, prediction/evaluation limit 1000. AP retains predictions down to
0.001 for curve integration. Whole-image 640 is a predeclared baseline.
Scoring remains COCO-style with a crowd approximation for ignored regions;
these are **not official VisDrone challenge scores**. Keep test results separate
from validation and do not tune the configuration after seeing them.


In [ ]:
import os, sys, pathlib, subprocess, json, csv, zipfile, io, base64, hashlib
assert pathlib.Path('/kaggle/working').exists(), 'Run this notebook on Kaggle.'
PROJECT = pathlib.Path('/kaggle/working/aerovision-lab')
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
os.environ['YOLO_CONFIG_DIR'] = str(PROJECT / '.state/ultralytics')
(PROJECT / '.state/ultralytics').mkdir(parents=True, exist_ok=True)


In [ ]:
bundle = base64.b64decode('UEsDBBQAAAAIAAAAIQAvl5U5fAEAAIICAAAOAAAAcHlwcm9qZWN0LnRvbWxVUcFq4zAUvOsrhM71w3IgZJeVoadedqFQ9rAEURT5pVErS6okp/HfL88ObXqc0TzNm3n7w+T80JS5VBw1y/g+uYyFK74XBeuUaoy+9Gq7E5qt2oOxbxgGrviNApa35xGrEYztU46vaKtmwYxISoM5nl1xMTTeHAQ7YyZATy1IaAUbsNjsUr2yT6Pxnj88/uV4SZjdiKEWfoyZG8zOeF5I0MQD+fABK1oaFZ8RmjTX0/pXrzYgJVkkDAMG664JJ1+z8XN1tii1gw103U7ccRGmMc29ktBt735tiHl03sePXsl2QfO/+z+/e7UlkGYbbbz21EELVNVnBRCXSMY3t+6aDXheVkhzxVJ7tfjm6XjsVQs/hGZ4STHXRRNDuCglQe5gsSecp1DdiER3HchvlmuRRbOv1r/fAKx3P0fjAt2KNoebSyZj38wLFji6MGj2ccJMJ9yLku1iswyse4ML7nlNWDQjJpl6WsslVL4GKJtm3gVsPIaXeuKKS9my/1BLAwQUAAAACAAAACEATCns5zIAAAAwAAAAGgAAAHNyYy9hZXJvdmlzaW9uL19faW5pdF9fLnB5U1JS8sxLSS1IzUtJzStRSEwtykzMUUhJLUlNLsnMz1NIrShILcrMTc0rKdZTUlLiAgBQSwMEFAAAAAgAAAAhAAZzVtg2AwAANwoAABUAAABzcmMvYWVyb3Zpc2lvbi9jbGkucHmdVl2L3TgMfc+vMH7yLZlMmZayLORhaGkplM6wbAtlGIJvotx4N7FdS8m0++sXO07ufPimH0+JHZ0j6Ui20jozsKpqRxodVBVTgzWOmNTakCRlNGbZsucOVjqEZf0PGr28G8xaz2Qldb3aLzTXkrosyxpo2SCVFrs/M8YYM1iAnpQzukCgBlo59iT4l6sPV9Xrq49v37+r3rz/i+cMyQnPUdR3jdixc8YLJEnA/evYk5P9d1I18t0uEIf4HCvXWItLdxgH0HQdvogGsHbK+sRKfgnOfFaojGYf5J7hIPuevbv+xKyy0CsNfGbFcc/KyF3Ipqlw3M8r9IRU8toMg9QNz5mDr6Ny0JR/uxFWeEDNEMEbU5NxPGcd9Lbk105pYhM4HwgyqZsQg9KtcUMoQQzDOrCsfELnt6VbYvWr8FnGxAU/O3PGEM8ZfbdQejlzFjUPK8EbSXKVMEVATiq9MChNR4I/nj/fwE2yT6IuNlEI0CRhLy9mVKsoIcQc5GrxhDZk+UCFRLFSQLCm7vBEIqdhajjgf0nUq5cbsL2kuks7Ow3ScoB09yWVgEnV3n5h5rUdN5T7mYI8Ad0Z9y+4tGwb6TvAMeSy2H40GtazIqljZFgvkQpLMWSYEt1Qm+HesYDp99ohgbsDdegIfwf6Ex2RQJHq4ZdBZgLXS7vg2t7I+/oXF6d12WiOhHmvBkVbRU4FN5IdN28kN2o8n0uo0N+AJ7kG+e2sgXQEL5ZrJoGDSfabYN93p72axktUd0bVgOUNv+tMqBL31fKHhe8Ndfz2noxh4yQj2j7oeKScL09OgOSL8oDLf5uppDsgK9kkHYo4osLDk6OIusX5NA9GLKyxYp1Zs4Vqj0YlW2bUPK7DUfXTvYBvFpzyIS8TPs5xv5WtxtaPNOH/D4pmHCyKe1ZilzOlG9D+Potq9I+8LyPtsXt/ZBfH0eax07gtfJ43PAy+2zxkfRMHxLr0Gq6LcMXdngpoRv5YDfJ2x5DCUjx75l0szAg/pol315HIt9uD4oX+myljfUPWs6s82GMpHnflzpd55ipjg4aImPCb+W6XZZlqWVX5cVJVwaqq/J9bVcXs59+47H9QSwMEFAAAAAgAAAAhAOPSpVW+AwAATQsAABwAAABzcmMvYWVyb3Zpc2lvbi9jb25maWRlbmNlLnB5vVVNj9s2EL37VwzYi9xoHTv9CLCIChRB0VOBPTS9CIJAUyObWJkkSCpeo+h/L4aiJCpWi80lAkRI1Hszw5k3I8bYx6dPD1p1N9AGLfdSnR6MlsqDww6Fl1pBa/UFHP+MDRiLjQy7LgelPXBQeIVfn8AJbXHHGNtsAl7oLvIdyIvR1sNH3SuPNocGW953nixF9K6R/KS081JM8Av34lyjtdq6zWbTYAvuimgyb3t/zpexyJPSFpsc/NmiO+uucUW23+1/ymG/O4QlPL4LS3j8gZYfaQmvP9Pyfrt93ABAYgYKcNp6bDKHPpv3t9sAlG3IQ4LXFri6ZbS7hw8FeFoO0GoLHqRKoNEXXZZLh/AX73r8jU6csT9ni5feeTgikct9DoeKRd8XfsJahhj/tiWTDauCHxv8UJpKFkCOVf8EiuAeT9reRpZIWCJhRZycmZ7bE3pHeddGO2ygSAuZddL5bX6/FciuN6GmxSiCbNgntzxxy5XSnoeasmrOTkwy353QZ0w6YfW1YTnskwwmMZY8Hrums1U7bgyqJuODz/GKMRE4yQqrKnhTwGEKz1B4idYWUZnUUYhRqqQsRP/C+ghKC7E8xL0Unibvg+23IxmkA350qPzQpCere9UMyYwaoWusWGlW82KimoYWqo+3OoBGUU2ER7AlO+oX0sQss0gbRHLsxTP6oCz/uNBCrPt2rQ8GLn0YndH32HVTNpNafwe/W8TmBg0K6cKMIfZZns5oH8IoWswqEEFY0CAdGLSCjtMkSgx2+rpK3cEfNIdAK4E0XVCB5884zDsCtvIF3W6yVOcgOu6cbCWNoxqKxSDLVvU6nrGauyvd+7IyoQ3G7zmU1Tade+W+mpzMGpikPEe3lF2oy2hkWZ8lcJZ+yAGr4JdiBt9jE12UE6wiLS7bjnaepVp2oNVXUlM5HOlVMRq0dTgkqXCQ1sid+2Z10C1jF5JG3EgZ5uQSQJImLyunE7JZgr3JoaVbkdHALJk3rMqntza8jWOJLMDDArowOB2znENU/IKUvgJCz92VwpvCm1n1U2pM0a5tq6JV99sxwCIN9B5lbGzNwht4Cxf+kh1y8AbeQGu293iLgnfdClitgNtD8Q6+hxQc38n6GmtuBG/oj95fMjEkdf71TRndfabh67L4ix9SNLPa17NUwlKvZcXEhj9sSczD/zOpG0XJIo1Vs06oe8Ypv0jHujyEVq1sUAksJil/48pepLC6/vr6Bi6PXMr4lMTtbKNDleyvWJhyW0xP/yWj4cmi760Ked78C1BLAwQUAAAACAAAACEAR72ufMQLAABDJwAAFgAAAHNyYy9hZXJvdmlzaW9uL2RhdGEucHm9WvuP3LYR/n3/igGLIpKt0+5dHnC3VoEWcQADSRw4D6C9HhZcibqlrSVVktpHFvu/F8OHXvs4n9FUP8RakZwZzgxnvo8XQsi3cisqSQuQij9yQSv4jetvlRQMqMpXfMM0UFHAmn5kUDDD1JoLrg3PQa9pVYFulpoZnU4mP8tG5WwOK2NqPZ9OH7lZNcs0l+tpkNm+3HxLDdXMTP7ulMCaKyXVHH6tjKLV3vBcA9VWMvwooam1UYyuIZdiw5TmUkAuCwZcA9uxvDGsSCeEkMmkVHINi0XZmEaxxQL4upbKABVCGmq4FHoy8d9WVK8qvgw/P2gpwruiopDr8KtRVcWXqWL/aZg24evvvC55xZzGmhqUFdT9RM2q1bOn68rN+unt92HG2zV9ZJPJj3//4c3PkMH9BACA1Kxg2ihOBUnCF1lXLPxa8nyfdz9zqsLrpltiVJN/7H4Ml9Ct4OLxZvx52ejwupZGotiHyQ9v379/9x4yIGeC2nShmrpQTRWrGNVMTwufVtPNLJ2lMzKZTApWQsEfmTYR+mpuXRTDzd9AGzW3mleQhZikekXvvv4miu3AlpuV9XAqayYiopYkBqqhdOvwKaWCZSXzj8AFcMNUVNH1sqBzKFPFaBHdzu6+gheA/8QJLAmJu8VWedrUBTUsslKcXsVMowSs0hXbedtjv5WaKs0WSm51ZNjOzHEXCWx5gVvjwiSwYvxxZewPu03T1BW7r7g2CeB/H5x+Qsh7p2ZDK16AoeqRGXfq+KOQihWg2CMmLso3K1jKHdOQV7yuWQFGglkx4JhOKZ4AFOqFJK2EDO4fErh/mARfVVwwdBUan+q64ga/6KjnFV6CkMbOTDEn6/4gPrkUhouGtR+V3KKmspLURJuwKLYKN6jNyUJtEUlIjCraaQ99xRUTkZLbGF7Dq6FSRblm8ButGvYGa0ZUkje7muWGFWA93lWwkrOq0HM4oNojcTHFZ5fAPoFtAqsEdC4VSyCnhj1KtYcMd3E//6YzZzdLYD+DDNZ0F83SWQK7OOl+7HtybxPY3+JMLiLnBZsRcQI7eAlbXNaOuPyIE9jDS1h1QngJu1t4ncFuBlKhvNcZ7GdPuH4pd+h6Z2uCEm6c4fiynw2ca7cMWQZWARcmCruPMUjRLIHb21GsfSKltK6ZKKKl3HUWs4qXYM0cynqdwe3Ibp+YQcyBhNkLXpD5aP0N3CZAlku5I3Pc4LGvUrMn8+JX8VHIrehiewhvIRn8AR8fl1CufBWLbMb6I66kNL3ihS/OEBxI1x8LrqKaKiaMzn5RDUuA7bg2C/nR/nR6sW8IumaQQUlCvt7Nbv9y8+2bX24OVt8x/Z3X7jhjT7CJKQ1M28WT3iHFGalVpKMYo4offYNKuV741wjn9SJbU4WC7WqsLQvdlCXfRQR1pzjaOzW1wvCULWbg4hEOwZgjSaCsGr3q7RIfW7GG/TNtVGXreEkOrsMcpwM5hq+ZbEz2zcxWecV0LYVmiTXXt4DtaQuwnvK22ZI3Gwxd7hBBwac1Cisr3So+7BUXrHiZ2Vp2YSLvkowV8GeIvrzzqoMF9pieGtCPR+e7ORx64qZTGGzmCD/wf1wIk0+jMxmDPo+fPGpvhetePfUeP87hgCL65deGUbG6ornPyK7LBwv+xevv2oTFQAdxrZQ/wbuaIZax/S9nAuEIFFyx3OBhp5WWoNgHlhsNXORyXVfMsNZEnY5371Wk6Evs0ON+d671rGuz720VzR0VF/wUQINiWBqiroYkYBTlYpHLRjisABm8ms0SRAPDr3f4VTNWhA9f3Z2pQL5MpIppWW2YB1CyMXXTLyEEYfyNOfS0H282h1bp8UYfUNfRFSBeQuRlTIGsqeAlHmREzCRu606vrvjUfIMjGCJHE+ZwcFIuVQvvsk5V4XhCihDamRKUQwaHdh1BW8ncuifpvuYVAlNN5mBhdm/EVlgcOBx7XwPlGX+vlTQylxWZA/nnu+/ftSBNrrnxTcPBsr8C29CqsUTDD2jIldy2IA75jAFZljznfbq1ZgjJdepR+LFFatbUBGxYbHuOiI0aFsouenECEdnQivQyJ+5FxG9tgSgaK/6gtbmuhv5uPdDFJDj8vht8uLfLHjAGBE8LmQ8U2BOUAHEgnswD8O/Pid0ObXYhdl0UXCVQ0SWr8BWyXhbYCZrANDijG7ILuqFTmZ/akvFptT9nkWK5VIVO+hSzRRILP9oC8BaEX6x4AyedrXzt/lCslsqwIhI2VwQmyGkZw/NLpm7JlOAcYfmFSCu5ZSqKUyYKjdZEJP1QP5J42Kp8fZzBa5+Gr11XcxLP9MczlfK9a/6sgIOrMOBhjt9KAnRDeUWXFYNDT3i/ceCTr6Rmotu4Y+rpe/tPhAUgTjXFWu8l+KMz2hJ6y+UILxJYs/WSIRQGJpo1U8gDnaIExkh4hN+w/EZufWxdfooGXLVqM3IM4cZzHbZYLPeG6ZANDpp4Nac4wiaSvVFw8KjtmlbpefhgmUmgqcG+VPPfT61qEy7dMMXLve8q/YcKsfBezLw72xbfpV4CZNo7JlMSp8rTwZSgq+9nD/ASSGp2xpX7/qPodoF8FTIYuKXTHacFw3uhiDSmvHl1o/njKH0usOMenQ9akqGHTsW40mNP9mmCSOXVWJLtFJ6PQ0dGIfMz7x3pORXbqQ0M6uwUawE5eGFf9DjWFw9HOESWjMIU7mKY+juL9FWJI5aLhhF/g+GGtqOpq+GE02jhc+o0fHoZ8OQ2uh5/7iGWNWK16MmM4SXcJk+s80cfOWeoAldXvHjhvPmEXKoYJXPYwgtYPWWCtriAzGF2eWbXJK+7Neoa59SVpFBi4hGxw5MVx77GYJ6f9z35tyDpB8mFk6zjBJjIJZK+zB2u04ifmuW739UoX44w+cT4ENzrwuOQsO8r020aY5Ts+b48z6U3mfs8vzJzDHRs/T0//zSiZ2jhEDx0tyXn0haIRZ74yS07nsNuHvD2kJsX3s7NZS4HoNrujItSjuCw++4g2byFP8Ph3nFEbNhDRsN5vjRxK+r+4COeQECV4ug6deKgTdedLaCPHzpxnV87plKScJWCmwt0pZf6+CUtmnWtI5xxJsnjq2LDrdhVyaNoXlUSiNNPjikWASThJSZeiXoRHWg6jIvfMfQdlxWnPMuRBrxnt/jxoO5D+j5YX1sYdCZxPOt4cI7mJVCxj/qLsdH1ZV8VhlTloYesThAjeVswYXhOK7dXe+XJhHEwXAPNldTaabRg1l5AWCd4f+ZSlPxxyBMRVJO55w8p1Ytaar6LemfVb3MeUnzqyVY3AU3vhi3n6gYxbzGXC56baJyuXo3zoGdi17iu5VK9pMKRVNOSLTCzIrfBxCLhxUe219l3tNLsYoJdofDnMzfMSoALjEZ2d1G2Z+5o8fCyY9HFZXDvMbjb+PqrV+O7jQS6W1f8CxT6eXTfQUg4J4AMuse8PU0MLBdvIIR0uYLv6On2byV4IY7TLcVBfu1pNDFMm5uCbcjVNH0z0grrRhtYMsxHvIjtpDzzjiYqSee6m81hdC/T2Z1579hLcShbu2/EeI1T1BGRKxx7OPNT+fDn3TJ8JoP+TPb8f2POz2LNp4nVEeYuDcItkMbAaV60f6WHpWxE0b+2eZrgPptAn3LhM9T2Oh/+DH77XG77DF57hdNSIf4YInudfI557VlCe5WZfiLz/FQOZvvYReDLi+x55CtkZPY0qn+acSHZyoaM+f7uAV6MWPT9lw9XDHIsLLvAwU6x+fDLUwTHuq+/XWi5SmbPgrsptfHMBlHNPOkAh6+yPrMY2jC6qzvPHZwhJ953KDHzi7xgzwSGoUcikPVpgDsK2Qn+7yVDdhb4d5g/u2/9k4D1iWWsLeTHG7bLqH8Mav5naP8PR/qfBsRah/U0DIspOg8Ldmb/7gHa/h9YC2tx5ltu6IfDLBp0SY+0dOb22lK6Ho/s4ePxjpJeLFrEfQ3UuklnNzrCt6ebRXuzU/BuM0VnFzA3vHgRdlYG1O7DSp6/Rw903S4m/wVQSwMEFAAAAAgAAAAhABcHFKG2BgAAwxEAAB0AAABzcmMvYWVyb3Zpc2lvbi9kaWFnbm9zdGljcy5weZ1XS4/jNhK++1cUmMNIM7La6ezswbMaYDeLBAMEmcYmOQmGQUslm2mKFEj6hYb/e1AkZUl29+4mPrRaxWI9P35FMca+dmi4E2o777RQDtAYbcDgQeDxE9TCOqEqB43RLXz/9fuv8M8nwAOXe+6EVjljbDbzi+t1s3d7g+s1iLbTxgFXSjuvZmezKPvdahX0Ky0lVn6131Bjw/fS1aJyQafjbifFpl9/4m43m81qbEAf0EjeJTyDTQY1umBqzQ3y4gcuLabLGQDAKYNzBscMdlAADyKSkZCkJN54sVAOjQ12oICWn5JFBq1QyQk+kIWTfx5TmPvFE0nSFN6PVc/wAXZknp67XvVMkjT1bmpUuhWKO22ggCO8hx2I5iYFQGkxLpJP+mcH80mM3ppBtzdqGvvDxIe3Pbx+hkUwvsgXsZYtd9Vu7RtvE8fNFp3NoDNIjaD+ZCC2ShusM6i0akSNqsJikT9+zEDofbHIP8ZqM8Z+qbTBuTY1hg2SWzvnR24QtgaxPgd/Qm2h0QYOwu65hFrwrdLWicqCVvLsgUUW98qrYw0FWHSJ4WqLiUTVR5rGunpHohFes1x5GTkY8gChwGrjsE4myT3juZC83dQcuiV0JbOUAltldAyorMWvZt/jySOlGVm9qsM/RtUZlH1sWjmh9ngVVrrtuBMbiRSs8IEKim9IVzQQMyzFqmQVd7jV5rwWNVtBUUwimCyG1Om3QesikgeHk3zF8nqSxvY2G32i/McBBFma9ae0+FkrTMc18e6EBaUd0CJwVf9f5mnj4AE+F4SqaQWvdckNtvqACe0ZvNPvWShqPXNdQA79UIoGuDon/y2MjT7dUYjvOEWyyD/67mz0ifoTD8IIDGPXcXXs3+Lrqs0oygG6Oe86VHVCMY5izfy2gv70aNd75SwU8ELCJdh9m3Qlo5cAD+/H45/CHjykXuhXhYKSqpX5aLIh/NVl5KNkjSKTQGfu2oZ0zD5BMRt5yaAcsDOgO56+wcoqUlCYN4ndbyy6pSd64pq240ZYrQbJDfvgibedRFv8PXbEj42nLz/1I+NLy7eYhce/DT/OvJYze0fET9Mol5rXNom+4QHYgcu80pXOaZmluUFerx2eXBKLH+tE1TclE2Tbn7wlmJJt9AktC0kbSvptJ9HOa35CB2INoRiPxkSKHvrkg5MPn1DJRhOXrQbc9a3g42BXPdZ4sKV4ix5QVclCMlXJSBhzqUZ+It8ISjREapDK7fdf3bLOaKcrLdkS2I+B+y1vce5xAl/0b59pdAB30IgT1qP2fvIk0t83+oHBssH2oMuWo40jjVbXaNkSXi5BGALVe9ftiRUHdFE7/PSbBxiykWLePtfCJHgS1q31c+CFa+3JRThGx52WSGfICUknaCj+aNS82cfrrLrFyyTIhr2Qw0s+MvkadqaEM1Iuu1cB0A1BGKy0qe0wQfvI/LYRAvz7BGQDY0yJoBXW+rMyuWZMto0x6g0HBK4ml5Cblf7sbNElo5UMSppRAyAmjqYDIyY74VxvKoP37+9TKe6zKsIj0oJPQzsuAy/jOdCyKZ/xPOKD6DYSMZ7vebhRr7DxYL1kncFKWKEDMfdS17EVPPhx/202lX64vjYdW92GWzKDFZfyL1hTY2uRBMp49FYlPVcTTvAnM+xmy2gmm65GaC2hfHlewiGUKYODr10uHLY2Sem+8exJwldvaA1VLnSFrS53RV8NvoaqfgO/8BZ7iNcWeGW0tf5020/QCGPdddIAXWSrnbaoYHOOxHXgUtSed8FffIXa5pPjM46hXPa2bk5PCKAIwyrXHarRvAhVeThwyeCB5kwjJK4DP6d5pdUBjUvYf378F5uivDb82BulCZjTn4DzqaI/5g5bH+qkpDdx3n1W0a7+OnWnWWnpv3ReCDtLYFK0nigb/0YAH2F9CWxr+JldymAz3GfujVJSucHKcbWVmNC32DmD+J3mv73SjBhcCoWFjyCDo6jdrnhMXzfmufNuiX5knE7B44JMz+HbxzTNiIz9xAyBvhvdwN+tVhd4CWL/afButcwfmwvLoBFShnDuPN33IjBi7EYP6f/RibDnzV78qbKxM0qpj+yNyt3aWmSwyAKGc78hg8dFGnNmG8mr51eQ+Ubdk79l8F06JQZflzgCl/Drk/+cpLn4wxPQbeyhHzUhbni43tMIUp/8TChehslwGV0nrvZ9sMedcHizOg09pGn5AZN4ofDj2ZTvqP/LxXeXeZzVv3fbmPagGW4b19l9NMJhGN5+9tf7trNJJNMMhKpRueIxTpl4547Lsz8AUEsDBBQAAAAIAAAAIQAJhacXFAwAACckAAAcAAAAc3JjL2Flcm92aXNpb24vZXhwZXJpbWVudC5wecUaa4/bNvL7/oo5BgHk1Kt4N03v6lYFgia9FmiboE/0DEPgSmObXYlUScprZ2//+2FISqL82FyvH05IZImcGQ7nzdEyxn6seVXBP9/9DFZzIYVcA5clFKpuuOY3FcKXb798e2nsvkLALa9aboWSKWPs4mKlVQ15vmptqzHPQdSN0ha4lMo6MHNxEcb8TyVu0hotL7nl3czvRsnuuam4XSldd++mvWm0KtCYbsSKGv26DbebStx0i77jdtOvtud1dXFxUeIKUG6FVrJGaZMJXH4BpSjs/AIAOkyrdLG5cCNW7/0UXYWqa2Ehi7hIiw0Wt7lqbdPapIeka8HWwrIpMI3by4Zrg/Ty9ZtXr9lyChZ3NvtJtzgFY0vUOouIvn7zy/c/f/ttT26SGqtFk0zcCO4KbCwkb398o7XS05idL3lVYfnOv7npyQn+WStvpbqTzE1ptK2WcN/DsWZvN0qyeS/+1I/kW9RGKJlMphFwgBmBh4cRoF+ezQMfMQle3PI1GjaPuKCrmZ+wk7TjovHy6K6V0tCAkJCwtrKaV3srCkNCdwrtH7aCsOlVtnWzp4dmX6hCWaUqwwaiDzHzbclzvuWiIh9gc28kKQ2nwgwzox2vm3YMuUabl7gVBeaS15jMJiBW50kBVgbheyXxgJOeahCFQ/YwD8HMnfd6iyShzZ0/eBBsVLExcxAyKEHUa/M+er/htthE78TrHIzV/tXvIBowiKUDhww+vvZjd0rfovarQAYzP6rRtIEW/NvtDLKwQeeLxOQJX3TqJR+PFNtB/Pb227feWb0XQuaoJEy30rAJPHfse5qrAJPiThhrkomLbVLZjrFezJoLg/CVqPCNA3W+lKzYD60EXmnk5R48kTnce6IPn0GxUcogcKnsBrVbGJSGy0tPPthWoeRKrCFzkWcIGqSmzFidOCPXaFS1xWQSGZRXXOZ/hmGnvszdh0Gnw8zdh0Gvucz/DMOkv4xuw1BQXxZ+h4lalVhlbK8qdXUl08ayYU41VtTiPeqMvSp5/Ws0VelZNktns6toCRTrDflDwfdubvZymJQ35pB5Xjc+Yg4BjRcbzL7ilYkGS7SoayGFsaI4QGi4FSgLPBJhUSmDea0MF0U2i3frR9KINVPwCrNZ+mIYWlWiqfQYrNHqdyxs5u0w2hmvMaNbpFcypFzdHm6lqZQ1B1swfIt5g1qoMouEWfNdXqLNXswC997UngyChFIYCivOcSpRCAtrzUuB0gIvirZuK5ei52A3CBr/aNFYLL0hgTCAO4+WOsoUjSGLs0aUWNl8nGaj8NUTdomAHCGaNJZri2Xe2oJCnKiRMt+KHhL29LfLp/Xl0/Knp1/Pn343f/rjv9jUw6xrB9G5yoO7x0EgrW9LoZNeziRSL6AGZUk1zlHQWLF7UtLDkHaoLmExUnqnhcWcUnlCk2nZ1o1JCGEKQpYobXY9mQLKQhF8xlq7uvxHiAKPBf4hDkUgGg3avEF+m9dYK73PjeUUxHwMJtFB5gXSoF7lhWqlRR3mnd9C5sJl4sMRhabYkXu+joIhmrYi4o5I6lOLBwo1zDiwHEaPULdU5kM0nz27v53D1qXy2ylsKZt7I0mFxZoCtljBLfwtA+bw2MPkROj3tFPnKqXQQQBo+YIZLJQsDVueFhVcekH+CQ15uk4tpCG9xTK/2VsMawwEyEWD4jrAoJsksP8cGO7It8lrvL1N/pqRNVrIEd49WThzCTisOplCL5X5oZQeJlNYVa3ZRC4T6kWPHaoNfzzA0/WGD/QmHhoqDsjgk49DzLKiwqNBtUVd8WYOq0pxV0qk10d1CBW1sxBkK1ELe1B3eGY9ByNff+45F07WHjaE0o5CH1HpsJOPJg8qmM7NzBxs21S4oAIJ0jQlQ0jY3UZV7gxA2yy71UxTEbdhD1tesVAKHR1LCtXsR+/b6/jVFbPADcjmkQIqqnRTeurA6FB3Hoq2HkPS+4eLsh4gJYvopkqxRmOjSRJHj9hopI37gk6svHhcheYKeycfYBaNvSxxyyaHFdsvvGrRF2vsTX86DWTq1li4QaDdKA0DlYsjBUM2flW6s4uOM+LJaZvmuNwn9DJweqhtF9MciJAe71Hev3OUO4YdMapVnxPbnmDHR8fh53A1I1ZGbH/eW/Mja/2Af7RCY0/pi4xIUWU8ovVF1kF0AjMNFpC5w3Vq+ArzSvGyK1956YPWUWAKUUSpPl4TnQWj8ztb+lmrW+eoLnIRVZMkDsMlZqfNB2eZXZT84HqiphMmRWQivWD+nS0XcxcvliRK9+QPXQdQnkRJ+Pd6wUTJlk6hmrTpgXzVcYAHWZiNJ6N2iINYcEeLE62TIGIFPNDM3dK0aGk8V0+cR0Kh1V0JGy7LiuoZYYDLvmQD3jRa7UTtvcEq+EWY15qCl1hLpbEEjWtazRd33eAHFBDA/msdSNzZXBDZmu+ShEeCPLv5CRUXK95WNptN4CO48pGDZK/unCA8E4OBUwGj7kbyCl4pSjOA0VUoaYVs/emwo3ujdgTsaNyoHalxjEVABbe4VnofsR2GxDHCOd2nvKFSctw36q7x4fDwEmUWpDk9DxMEkB2I4zxGtytC6p69ks7j3NyoXXajduchuEZOEIvrJTwj8S5ePEJPGGfK8eEmvsY9n9MjnZ19lAVzWVOoITcJBdfaupRkXJynvJqWiA09JE5PPVShkVv8Rpa4O1VHUxUVqptg4qG/4M8bVBLJ7gQ3nPLiUsolvf/hGBVWzc2GX7/8hM1DWu25iUBrLsUKjT2CDf7cAwQ/jhtzWllVqIrNgUVt31fvpiHejOMKOcZBPPnM+Z5arUQheDWEHVMoLeQ6Ohczg9YKuaYq9Nj2T/Y46KJsmNFtPByqxiz8jifpSHHYjXCLqDabpX8fD57pmcSH7vA7noxzZxa/jMFcKM3cfZiIddD5LcnFKDoeJ6IcK5hqBeqdhoalz0RH9UZUAqg7447OJO4puNMPpbbFchr+96BP4Feua2gbQF5sPEElKbfwwrauKuRrnJLdG1FiIOqziONCaOOcbHudipoyhHOazvRCnoTnoUJ7HtLlYrZcsJWofJ+ULYN3OSWtAlFhXN09DrVD3+57Zb9SrSx9kXOGbNStcXWnK+KqqV9h6rY79bZH8qpoo8GgwNsPkL0EG5keGsRAvqBIX+Z9ETIbZRyNhdLlUEqMt+TG/owIPblz8gsy9FRPyvAxOZ6ifUj6fD+bakovK3eEL6g/fpwpB3Szl8VGKyneYwi/vdt8oN1xoFcKRVOQkPWKPlo2aD7Y81/U/Ij6/0VAwbu7EiN5pNkxgWdwNZvNxgR8VOjw5cHkyJ4/yqBCmUSynkSnhbF/qjuT4s4STaeHoUgJluWqDXj2rJkMX3RiyofCpJU9rxN4CtcvIctgdiw03wlZsXvS7MMc7iO8h+fuze9m8hC8cEq7zO7DLh7YcSdk3LvpaKcRt483cUgYZ1s3YXsEM94NNbg9dcioRqHq/AcM1HrIcbftCC8qho7no+Lovosv814qcaE7P1X8TmFU5s5hsXw4t9JxgeWY9yd3RT2Rrt+QrO00wpwCo8qTnUBK6Ut1bVJRr79xR7YobZ6HrvnuNfo0eDWFq9l0lMGXJxDDU+x5w2TfUo+nn8BPGySnkyXXJZi2rrnew4brsvAHfs9EdjWbOet/9e6zrkhUre7PcxHBDuPFbAamEgW1UqSxyEtQK2/17hs+1MJUyF2z+/IKNsjpoIhDpm40Fu7LaGh9DHtcsH6OLf3J32txBTVymW95JcqER/0MutwoZMAXnILBbJBg1ER0Pb3EgaZEK5m4Dq8fMOI9Dp9Ah4W9QBah7lku6JfO0cdxnTdZxGG/i8V8Cv7fbArXy6iW8kgvZ6fRZo+j5Yb+duIDK16dQNVY8Ko6Rj9Ug4ejnsVZUryhj0N5UXFjsvHHdLqKBfNJmzq9Z9m87XbYNeILCsIo2xo1GfQJnx+HkujLOV3kBbLY583LWV6bzCtdNmmNpeAyCakq/tI5wvr05QirQV2gpJzcYU7h05eH2G57rrUYr8f7mH8APgSWjFKBj8zHZZifDYni1HzObT7E/5wyyChXRjX+iRRCfyMRutEfyBzOBf6LDwB0HX0EOOlAI2Jnev4e8eI/UEsDBBQAAAAIAAAAIQD1H+UfTwkAAGwaAAAfAAAAc3JjL2Flcm92aXNpb24vZXhwb3J0X3ZlcmlmeS5webVZW6/buBF+P79iqieplbW2Ty6b07IPW2wWBbZJ0DZAC0MQeKSxzUYiBZLyZYv+92JI6mbrLJAUNYJjazgckjMfv5lRoij606fP8P7T4xY+fvjwD8BLq7SFs7BH0Py8Up1tOwtcVlCqplFy1SpjW61KNEbIA5RHLL+YLIqih4e9Vg0Uxb6zncaiANE4Y1xKZbkVSpqHhyD7l1Gy/21Fg35uy+2xFs/9xE/cHoPVrOKW9/JKHNDYfkDwg1TGitL04+qEuubtw8NDhXuo0GJJqxf8oBEblDbWuEeNssQUSi4rUXGLKQjVFY2QbJ29e5eCKZXGgltVs3W2Xm+SpwcAgCiK/sJteYTdZZNeN+llm163qVNOy5obk4N3mnFeVJ2FCluUFXlLyTAISleondvIaCc7gxUwMGhjzeUB4xplPOwtSRKn1wgpmq4phOpSaPjFPfiNVlhbDgw22TqFdbb2+rRTZ9g/75UGrc4gJAwu8Meiz7O6AIOdVufdOk9JcbcJ39scVjAdeOwFmzwfDJRHJUo0ZES4tQStFA4n9iCkHc+0E/nudZ4AY05Otl7nSf4wWKPgCdXFIni+/yh7RA1sjNxOjFugj0bbadnDIH5WlxR2bpbbvv+1GX65s90Ou/P1mnky7uoZjQVGzo/DcVP4glfmQlLhnne1ZR+URB8x+oi9nyUM0MD8NKWSVsgOB+GJ1x0Cc0enaaOdPsT82cT7WnHvtFd5Aivwz6NLaGZOY+P0CXjoAELGMzi5dSfai+iiYy+M0Mlry2dH9uf4I+uvlSMRb+cPbHK75u7wYMk0NuqEN+efIvp3DDYPk2BXorTxoNlyY7BiA/wZ0HUaIO9AN79g6TB30CpK1UnL5jNHvWHuRG/JXthEv5nJwOh95sM3kcwM3PmbLcXATUgC651Qi/01PqM4HK15clyagmj4Ac0T1MLYHYkI746Qeg28tFharApz5NvXb57AWB3uX2DX8rSdPsquaa/ADch2KlZSXm6fdSeJ60lZaTsdtUqXR89QROpdbTWvr1NO/+fHnz8uKrjMkPHuQMTea/+M1qL+QV2Wp3RW1CaTzWBdKlk0/FKYrm01ZTYlx5kZ+USLqX2UJ6GVJJFnBsLbkJl6rzuY3Tg0heiz7GVgVKdL9Em0VUJanw58SLLmSyV03HKN0hr2d90hxUcYW6gv7tFfDee8zKAtZNcU9qiRVybeTgcbLjteFwaxil+FEZ/qXXKg0I+Xh1wdG6uHYyTkAaUn98v5RumGWxZRZKMRrT6qB/ML22y/X8/lz3QF2GYurPAkSmRR2XY3Zqqr5I0o/clnI0de79l7XpsbuRFNW4v9dWlMtQYt27ydS2VjbpW9e/zfEG9zJMiEUGsr9rykJBBy+XcQcdTqJAg3q31X16t9+7jNnGe8mf3g7kyjUfUJ4wR+wwZjo3SkQ79oVqr2uo376ekwxW+Q1sgcflD776JRFdYugINq0G1dEUYb1zb7m8f5Ry+MZyqZkFbzQrVTSAGD7a0S6nslz8vGmw+L/VkGAg2rzreXOu0i2GXhO4WWnFqhNmxHdeqPFyw7GvoU5FHudz3QM7Al9DqHZJ5ikwxPvI6TbN8ZjE+on5VBj4AkK9su+MFVcq6SmdF/JnmDpl/UQYPdpJ4JNbDJ73hC6P2xWPBRdkBbDGedalJ0ixNq0mIu1EX/WBRztUCuo7a2y8rBL4GP2C0/zdcOSoHXxoCN1QrlGVG68r4oVYU3Uxy1FMVe1FgUydwJVpWqZmMHEpoPw6WwV0+KVDHvxQUrKicosxKkfBb7PUhlgW4b8LLsNC+voDT89OkzPKMsjw3XXyaE0moc2xbmYjahKahdxnhWl4J3VgWyNVaLCtnjNoVS1Uqz6K8//RBRa8BrZJFD1OMWvoPt69fRtITg58KqGjWXJRq/mO4bifUmhUlXMdnhrLPys0ol92ydPa5dd8LW2VtX9xcVUdl6vaYnWx5dDfFC2zIu4P3GdnlfKoww3kUB8VEODG7wPrQO1J9RRR/KiLlhqshP20w0xAPuBpL6pP6kBsApLtbBmguD8F7U+EHZ96qT1Y9aq2BkUpwZ8YvLWUOKjyWeCXQtstjHkv4mKSwFMondFpj7O5rlWvMrMJBtxo2ryA+d6owTx2HN3VMK9O9ptckzq7k0rTIYb1OgJZMdHSlPobLXFplsPd88bhMPj9CQ0ceiNIoaGJ+bqcYgBm2vsVtv3JWx3BEM3eusRb33lSbqQFLuNlOn7g2JnmNdBphmkuC5rrbT4MZ+I8lLavRjt85d3IyQxhKayRtdbVOIbdfWmLpSMkkAa4NhzkIl3RCTxgvngJU/ZQK/BcLzV519qLZd4+ypVHcypkik8O8puwpJHXmc7Na5w/STD/l/SLBQ0X/9fjU/D07zeSTzMZ1gl58LakxIj58zB1mqEIdlg4j6pGel6pjQWNdlTTDT/Dx7V+HoZIOrV4FMNrh6nNy1vl8jE8+GZsNqnJ5QTH91Dy6c5MdpMIl1yC/zAuq+dA4YycpaSYwTIk+5dwWCGblseH6bgixZoJ85u82iQ59Jhh5Et24efyltw5aXNjmzcncVR4dQ/RUn/8Mpxp059N1sl1ifiojlt1RkJO0PMqXBUunqrvagj8u0BHFGzOnAflMjz/Kzo9eb6p3uSmBUut2eljwwblR7SGPF+p/3CuQL/mxCj+uQmdF7hMTB0CNVGJfNCXEj9m6aB3VZMuVo+VX+LRZ9qnzB5qunb7I5xNEwiux8sGy7YkqKbPpwr+oqsMawKS/dtinTFO7TcpRnvKUXjrFHyQRqml60jbCASPPzU5QOYUwhGvdPA3SEXeQjHOUp7OvOHCetZ7+yuUp7RCtKDxtfRez83dXy4IlIc1mpJgvvyAotD0MvSsWFJx4hYRdvUnhM4c2rtfuTpBAks/QeZO/e9KL86S6fa3kIy8bO/EJ6/vpM2r+gmyTS5USeLE5z3/d51YmX06ob+r9lPZ+53BovJa4Xo9wjbX4F7jhpkVWW+MSh1PPJN2bAe3v3FPRCUgzXfW7hNq/0vujvxCwfuj37WzipLWUFvK5jPZnk3sP7t/AvuXbRwMi3Ue7keje5sVH+a0v07DB9vRGPbzCmjVxG/z0TJdlZC4uFxYuNSZJVXdOa2BtMQciKmtxtkgLKUtF/cLCos/vV91HvK/dq1us//BdQSwMEFAAAAAgAAAAhAEbuRGucAwAAwggAABcAAABzcmMvYWVyb3Zpc2lvbi90aWxlcy5weY1VTc+jNhC+51eMOJl9HUR2b2jZ2+6x6qHqBaHI4AlYBZvaJoG36n+vbPMRsm/VRkrAM49nnvlMFEXfp6ETtbBgRYfQoOrR6hmY5FB3zJgzezCNwEcHYxbBjMOg0RihZBJF0el006qH6/U22lHj9QqiH5S2wKRUllmhpDmdThxvYCzT1pAOZWPbDIS0FIx4x+VV3VF3bMjg1ilmYzh/g04YWwhpy+wEACBuEC7D1xxSUNpf3w5SWUjdabEEX+ESLrqPZsIg/M66Eb9rrTSJflVGWHFH4KJH6QIyPu5XG6Dxz1Fo5FHsrXXMWMihZxNJ6cro7LkEgLE4LIALBa1GyYln+gnIBc6r7TgOcI121BKM0hY5MWhJ8Ukz2aA375y9wYV6q3EQlHG85PQhJFcPQx6CbzltUTSt/f8JtuPQ4ZLihUuxpY1MFGYKvZBkgjdvjoL3FgfpvEmD3yUo97kpDTMIuVY+AAKnjc4RPj3BvZcP0eUS/aCRi9oSL+wVx46GRulZg3STZmCsXjWNeQ/58GfX9U/HY4qCrFbydhAINR7OHO+ifnbSs+nKMVQAcviSpvQULy0chsMqXbde4KfHH+/Cz5QazIqqmK1b5FfZm5MHtxQekIfwEtOyAYvscxlYancxh8I1TUrhQaGNSzcyLgOQ5xA9WtVhBNgZ3PuGOqMuCy8prtSEhoKplXZPvwvQOyjp8j1tRUspzCmF6UJhvrgKejZPs4dm7PzMuBolh7KtHx9UMafZfKEwpdl0KekLoDHvuf89KlyBcvfzgldjLtR4FIZa5eFxVC1Vy5fnUXlHXSmD+Q/Wmad7cZGGNPiUQb5EmvjsbQpxgwqEgV+URLeoOpSkil1N0uw1EivkiJtwmqcZcqgS95JwtKxuSZzUnZJI9slx2iKjUKQUPpclvOUwpT9rLxS+BO28az3VhA0DSk4cdDcbir/qqsTleCOxw5beeMJ1ZufaKdmsaHHzS9q7fO6OsHNKv0yJ753/aEI/MEnNLPGQmD5JAvogWi5uJLaRcI3Po53KH+g399PgkY85UNdecZEtzRKawE3WT7bWJOrGLXjC0dQouZBN/pse8dWEVo8wZNtwVWpavFNwf8CN0n6nvothHyBPsnAey6QeRhInVrndTuKV+b8ol3A+1Hrry97yfXQc89x53YuoHlsH/BWtRK+CR9lGm0JUVWqKMih2W3AOht3LnJYUIk84ygLxvw//kc7LoUv+AVBLAQIUABQAAAAIAAAAIQAvl5U5fAEAAIICAAAOAAAAAAAAAAAAAACAAQAAAABweXByb2plY3QudG9tbFBLAQIUABQAAAAIAAAAIQBMKeznMgAAADAAAAAaAAAAAAAAAAAAAACAAagBAABzcmMvYWVyb3Zpc2lvbi9fX2luaXRfXy5weVBLAQIUABQAAAAIAAAAIQAGc1bYNgMAADcKAAAVAAAAAAAAAAAAAACAARICAABzcmMvYWVyb3Zpc2lvbi9jbGkucHlQSwECFAAUAAAACAAAACEA49KlVb4DAABNCwAAHAAAAAAAAAAAAAAAgAF7BQAAc3JjL2Flcm92aXNpb24vY29uZmlkZW5jZS5weVBLAQIUABQAAAAIAAAAIQBHva58xAsAAEMnAAAWAAAAAAAAAAAAAACAAXMJAABzcmMvYWVyb3Zpc2lvbi9kYXRhLnB5UEsBAhQAFAAAAAgAAAAhABcHFKG2BgAAwxEAAB0AAAAAAAAAAAAAAIABaxUAAHNyYy9hZXJvdmlzaW9uL2RpYWdub3N0aWNzLnB5UEsBAhQAFAAAAAgAAAAhAAmFpxcUDAAAJyQAABwAAAAAAAAAAAAAAIABXBwAAHNyYy9hZXJvdmlzaW9uL2V4cGVyaW1lbnQucHlQSwECFAAUAAAACAAAACEA9R/lH08JAABsGgAAHwAAAAAAAAAAAAAAgAGqKAAAc3JjL2Flcm92aXNpb24vZXhwb3J0X3ZlcmlmeS5weVBLAQIUABQAAAAIAAAAIQBG7kRrnAMAAMIIAAAXAAAAAAAAAAAAAACAATYyAABzcmMvYWVyb3Zpc2lvbi90aWxlcy5weVBLBQYAAAAACQAJAHwCAAAHNgAAAAA=')
assert hashlib.sha256(bundle).hexdigest() == 'fb75ff64f62ee72f2400a211b94742a4c50d0b1206be03806a5b2b7a2dae7e22'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        assert (PROJECT / member).resolve().is_relative_to(PROJECT)
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
# Editable-install path hooks are not reloaded by an already-running kernel.
sys.path.insert(0, str(PROJECT / 'src'))
# Save the source that produced the run alongside its outputs.
(PROJECT / 'source-fb75ff64f62e.zip').write_bytes(bundle)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator, then rerun this cell.'
print('GPU:', torch.cuda.get_device_name(0))
EXPECTED = '9ea4e947efe74f75703ab2c5a32bf5a6af70124945a6ee9bd366c7cd83172840'
BEST = PROJECT / 'full-model-best.pt'
original = None
INPUT = pathlib.Path('/kaggle/input')
for archive_path in INPUT.rglob('*.zip'):
    if not zipfile.is_zipfile(archive_path):
        continue
    with zipfile.ZipFile(archive_path) as archive:
        candidates = [n for n in archive.namelist() if n.endswith('/weights/best.pt')]
        for name in candidates:
            payload = archive.read(name)
            if hashlib.sha256(payload).hexdigest() == EXPECTED:
                BEST.write_bytes(payload)
                original = json.loads(archive.read('full-data-manifest.json'))
                break
    if original is not None:
        break
# Kaggle may unpack uploaded ZIPs automatically.
if original is None:
    for checkpoint in INPUT.rglob('best.pt'):
        payload = checkpoint.read_bytes()
        if hashlib.sha256(payload).hexdigest() == EXPECTED:
            BEST.write_bytes(payload)
            for manifest in INPUT.rglob('full-data-manifest.json'):
                candidate = json.loads(manifest.read_text())
                if len(candidate.get('splits', {}).get('val', [])) == 548:
                    original = candidate
                    break
            break
assert original is not None, 'Attach your full-data results ZIP as a private Dataset input.'
print('Verified full-data checkpoint.')


In [ ]:
from aerovision.data import prepare_validation, digest
from datetime import datetime, timezone
TEST_DATA = prepare_validation(PROJECT / 'data', count=1610, split='test-dev')
TEST_ROOT = TEST_DATA.parent
test_manifest = json.loads((TEST_ROOT / 'manifest.json').read_text())
assert len(test_manifest['splits']['test-dev']) == 1610
seen = {r['sha256'] for split in original['splits'].values() for r in split}
assert not any(r['sha256'] in seen for r in test_manifest['splits']['test-dev']), 'Identical train/validation image found in test-dev.'
FINAL = PROJECT / 'runs/final-testdev'
FINAL.mkdir(parents=True, exist_ok=True)
plan = dict(weights_sha256=EXPECTED, candidate='whole-1280', baseline='whole-640',
    confidence=0.30, iou=0.7, max_det=1000, eval_max_det=1000,
    ap_confidence=0.001, split='test-dev', count=1610)
plan_path = FINAL / 'frozen-plan.json'
if plan_path.exists():
    assert json.loads(plan_path.read_text())['settings'] == plan
else:
    plan_path.write_text(json.dumps(dict(created_utc=datetime.now(timezone.utc).isoformat(), settings=plan), indent=2))
print('Verified 1610 held-out images, with no exact image hashes shared with train/validation.')

def run_evaluation(label, *args):
    import re
    command = [sys.executable, '-u', '-m', 'aerovision.cli', *map(str, args)]
    log_path = PROJECT / (label + '.log')
    print(label + ': starting', flush=True)
    with log_path.open('w') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                log.write(line)
                if 'images, cap=' in line:
                    progress = re.search(r'([0-9]+)/([0-9]+) images', line)
                    if progress and (int(progress[1]) % 100 == 0 or progress[1] == progress[2]):
                        print(line.strip(), flush=True)
            code = process.wait()
    if code:
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[-20:]))
        raise RuntimeError(label + ' failed; see the saved log.')
    print(label + ': complete', flush=True)


In [ ]:
from aerovision.confidence import sweep
truth = json.loads((TEST_ROOT / 'test-dev.coco.json').read_text())
ignored = json.loads((TEST_ROOT / 'test-dev.ignored.json').read_text())
summary = []
DESTINATIONS = []
for label, size in [('whole-640', 640), ('whole-1280', 1280)]:
    destination = FINAL / label
    report_path = destination / 'comparison.json'
    if report_path.exists():
        report = json.loads(report_path.read_text())
        assert report['weights_sha256'] == EXPECTED
        assert report['manifest_sha256'] == digest(TEST_ROOT / 'manifest.json')
        assert report['settings']['split'] == 'test-dev' and report['settings']['imgsz'] == size
        assert report['settings']['max_det'] == 1000 and report['settings']['eval_max_det'] == 1000
        assert len(report['image_ids']) == 1610
        print(label + ': reusing saved evaluation')
    else:
        run_evaluation(label, 'compare', '--data', TEST_DATA, '--weights', BEST,
            '--mode', 'whole', '--split', 'test-dev', '--imgsz', size,
            '--max-det', 1000, '--eval-max-det', 1000, '--output', destination, '--device', '0')
        report = json.loads(report_path.read_text())
    predictions = json.loads((destination / 'whole.predictions.json').read_text())
    # One predetermined confidence, not a threshold sweep on test-dev.
    operating = sweep(truth, predictions, ignored, thresholds=[0.30])[0]
    (destination / 'operating-point.json').write_text(json.dumps(operating, indent=2))
    metrics = report['modes']['whole']
    summary.append(dict(condition=label, AP=metrics['ap'], AP50=metrics['ap50'],
        AP_small=metrics['ap_small'], AR_small=metrics['recall_small'],
        p50_ms=metrics['latency_p50_ms'], precision_at_030=operating['precision'],
        recall_at_030=operating['recall'], micro_F1_at_030=operating['micro_f1'],
        macro_F1_at_030=operating['macro_f1']))
    DESTINATIONS.append(destination)
    print(label, summary[-1], flush=True)
with (FINAL / 'summary.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=summary[0].keys())
    writer.writeheader()
    writer.writerows(summary)


In [ ]:
export = pathlib.Path('/kaggle/working/aerovision-testdev-results.zip')
with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in FINAL.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(FINAL))
    for name in ['manifest.json', 'test-dev.coco.json', 'test-dev.ignored.json']:
        archive.write(TEST_ROOT / name, 'inputs/' + name)
    for log in PROJECT.glob('whole-*.log'):
        archive.write(log, 'logs/' + log.name)
print('Download from Kaggle files/output panel:', export)
print('The ZIP includes predictions and annotations so review can continue on CPU.')
